**[Course Home](../README.md) | [Python Lab](../python/02_source_coding_kraft_and_huffman.ipynb) | [Prev: Ch 01: Quantifying Surprise](01_self_information_and_shannon_entropy.ipynb) | [Next: Ch 03: Relative Entropy & KL](03_relative_entropy_and_kl_divergence.ipynb)**

*Part I: Foundations of Uncertainty & Optimal Description — Chapter 02 of 11*

---

# Chapter 2: The Geometry of Compression — Kraft Inequality & Huffman Coding
### *Prefix Trees, The Economy of Codewords, and*
### *Shannon's Source Coding Theorem*

---

## 1. The Core Question: Why Can't All Messages Be Short?

Every computer science student has, at some point, entertained an enticing
thought:
> *"What if I write a lossless compression program that compresses any file by*
> *just 10%? Then, I could run the program recursively on its own output,*
> *compressing a 1-terabyte hard drive down to 900 GB, then 810 GB, until the*
> *entire drive fits into a single byte!"*

Can such a universal compression algorithm exist? **No.**

### The Pigeonhole Counting Proof
Consider all possible files of length $B$ bits. There are exactly $2^B$ such
files (for example, $2^8 = 256$ distinct 1-byte files).

Now suppose your algorithm promises to compress **every** $B$-bit file into a
strictly shorter bitstring of length at most $B - 1$ bits. How many possible
compressed outputs exist?

Summing all possible bitstrings from length $0$ up to $B - 1$:
$$\sum_{k=0}^{B-1} 2^k = 2^0 + 2^1 + 2^2 + \cdots + 2^{B-1} = 2^B - 1$$

There are $2^B$ distinct input files, but only $2^B - 1$ possible output slots.
By the **Pigeonhole Principle**, if you place $2^B$ pigeons into $2^B - 1$
pigeonholes, at least one hole must contain **two or more pigeons**.

This means at least two distinct original files—say, a photo of your dog and
an executable binary—must map to the **exact same compressed bitstring**. When
the decompressor encounters that bitstring, it has no way to determine which
original file to reconstruct. Perfect lossless decompression is destroyed.

```
                  THE PIGEONHOLE COLLISION THEOREM
                  
     Original Files of Length B          Compressed Slots (Length <= B-1)
     Total Count: 2^B                    Total Count: 2^B - 1
     
     +-----------------------+           +-----------------------+
     | File 1: "00...0"      | --------> | Slot 1: "" (0 bits)   |
     | File 2: "00...1"      | -----\    | Slot 2: "0"           |
     | File 3: "00...2"      |       \-> | Slot 3: "1"           |
     | ...                   |           | ...                   |
     | File 2^B: "11...1"    | --------> | Slot 2^B - 1: "11..0" |
     +-----------------------+           +-----------------------+
           2^B Inputs                         2^B - 1 Slots
           
          COLLISION IS MATHEMATICALLY GUARANTEED: 2^B > 2^B - 1
```

### The "Universal Compressor" Scam
If a software vendor claims to compress arbitrary data recursively:
* Ask them to run their software repeatedly on its own output.
* If every file could shrink by even 1 bit on each pass, then after $8 \times
  10^{12}$ iterations, an 8 TB drive would shrink to a 0-bit empty file.
* But an empty file conveys exactly $0$ bits of information—it cannot expand
  back into 8 terabytes of diverse user data.

### The Statistical Trade-off: Compression as Probability Arbitrage
Lossless compression is not magic, nor does it violate the laws of
mathematics. **Compression is an arbitrage on non-uniform probability
distributions.**

To make frequently occurring messages short, you must make rare messages
longer:
* In English prose, the letter `e` occurs $\approx 12.7\%$ of the time, while
  `z` occurs $\approx 0.07\%$ of the time.
* If you assign `e` a 3-bit codeword and `z` a 12-bit codeword, the average
  file size plummets because you pay the 12-bit penalty almost never, while
  reaping the 3-bit discount on almost every character.
* If someone feeds you completely uniform white noise—where every byte from
  `0x00` to `0xFF` appears with identical probability $1/256$—lossless
  compression is impossible. Any attempt to compress uniform noise will, on
  average, leave the file unchanged or make it slightly larger.

### Chapter Roadmap
In this chapter, we formalize the mathematics of optimal description:
1. **Prefix-Free Codes**: How to stream variable-length symbols without
   transmitting separator commas or delimiter bits.
2. **The Kraft-McMillan Inequality**: The fundamental budget constraint
   $\sum 2^{-l_i} \le 1$ that governs all uniquely decodable codes.
3. **Shannon's Source Coding Theorem**: The formal proof that source entropy
   $H(X)$ is the exact, unyielding speed limit of lossless compression.
4. **Huffman Coding**: The greedy, bottom-up tree algorithm that constructs
   provably optimal integer prefix codes.
5. **The Fractional Bit Problem & Beyond**: Why Huffman falls short on skewed
   distributions, and how modern systems bridge the gap with Arithmetic
   Coding and Asymmetric Numeral Systems (ANS).

---

## 2. Prefix-Free Codes: The Binary Tree Mental Model

Suppose you want to transmit a stream of letters using variable-length binary
codes:
* `A = 0`
* `B = 1`
* `C = 01`
* `D = 10`

Now the sender transmits the bits: `010`.
How should the receiver decode this message?
* Is it `A` followed by `D` (`0` + `10`)?
* Is it `C` followed by `B` (`01` + `1`)?
* Is it `A` followed by `B` followed by `A` (`0` + `1` + `0`)?

The receiver cannot tell! This code is **ambiguous** and invalid for streaming.

### Delimiters Waste Bandwidth
One naive solution is inserting a separator character (a comma or space)
between codewords: `0,10` vs `01,1`.

However, adding a delimiter introduces a **third symbol** into your binary
alphabet ($\{0, 1, \text{comma}\}$). In physical hardware, transmitting three
distinct voltage states requires higher signal-to-noise ratio or an extra bit
per symbol.
* Morse code appears to use binary dots and dashes, but it actually relies on a
  **ternary alphabet**: short mark (dot), long mark (dash), and inter-element
  silence (pause). Without timed pauses, Morse code is completely
  indecipherable.

Can we design a code that is **self-punctuating**—one where no delimiters are
needed, yet the receiver never experiences ambiguity?

### Instantaneous Prefix-Free Codes
A code is **prefix-free** (often called simply a **prefix code**) if **no
codeword is a prefix of any other codeword**.

Prefix-free codes possess a crucial engineering property: they are
**instantaneously decodable**. The receiver can decode each symbol the exact
microsecond the final bit of its codeword arrives over the wire. There is zero
need to buffer future bits or wait for lookahead tokens.

```
                          PREFIX VS. NON-PREFIX PARSING
                          
     Non-Prefix Code: { A: 0, B: 01 }
     Incoming Stream: 0 1 ...
                      ^
                      Receiver reads '0': Can it emit 'A'?
                      NO! It must freeze and look ahead. If the next bit is '1',
                      the true symbol was 'B'. Decoding requires buffering.
                      
     Prefix-Free Code: { A: 0, B: 10, C: 110, D: 111 }
     Incoming Stream: 1 1 0 0 1 0 ...
                      \___/ ^ \_/
                        C   A  B   --> Instantaneous, zero-lookahead decoding!
```

### The Binary Tree Mental Model & The "Scorched Earth" Property
There is an exact one-to-one correspondence between binary prefix codes and
**binary trees**:
* The root represents the empty sequence before transmission begins.
* Taking a left branch corresponds to receiving a `0`.
* Taking a right branch corresponds to receiving a `1`.
* Every codeword corresponds to a **leaf node** on the tree.

```
                         THE BINARY PREFIX TREE
                         
                                ( Root )
                                /      \\
                            0  /        \\ 1
                              /          \\
                           [ A ]        ( Node )
                          (Leaf)        /      \\
                                    0  /        \\ 1
                                      /          \\
                                   [ B ]        ( Node )
                                  (Leaf)        /      \\
                                            0  /        \\ 1
                                              /          \\
                                           [ C ]        [ D ]
                                          (Leaf)       (Leaf)
```

### The "Scorched Earth" Property of Prefix Codes
Why does the prefix-free condition forbid internal nodes from being codewords?

Notice what happens if you designate node `[ A ]` at depth 1 as codeword `0`:
* If `0` is a valid symbol, no subsequent bit sequence beginning with `0` can
  ever be assigned to another symbol.
* By declaring `A = 0`, **you have scorched and destroyed the entire infinite
  subtree beneath that node!**
* Neither `00`, `01`, `000`, `0110`, nor any infinite extension starting with
  `0` can ever be used again.

Every time you choose a short codeword, you buy low transmission latency for
that symbol, but you pay for it by incinerating an enormous swath of downstream
codeword real estate.

---

## 3. The Kraft-McMillan Inequality: The Codeword Real Estate Budget

How many codewords of various lengths can you pack together into a prefix-free
code before you run out of tree space?

This question was answered by Leon Kraft in his 1949 MIT master's thesis.

### First-Principles Derivation of Kraft's Inequality
Let $\mathcal{C} = \{c_1, c_2, \dots, c_K\}$ be a prefix-free code with
codeword lengths $l_1, l_2, \dots, l_K$. Let $l_{\max} = \max_i l_i$ be the
length of the longest codeword in the set.

Imagine extending the binary tree down to a uniform maximum depth $l_{\max}$.
A complete binary tree of depth $l_{\max}$ has exactly $2^{l_{\max}}$ leaves.

Now consider a codeword $c_i$ of length $l_i \le l_{\max}$ in the code:
1. Codeword $c_i$ sits at depth $l_i$ in the tree.
2. If we were to extend branches downward from $c_i$ all the way to depth
   $l_{\max}$, that codeword would spawn an entire subtree containing:
   $$2^{l_{\max} - l_i} \text{ leaves at depth } l_{\max}$$
3. Because $\mathcal{C}$ is prefix-free, no codeword can be an ancestor or
   descendant of any other codeword.
4. Therefore, the sets of depth-$l_{\max}$ leaves dominated by distinct
   codewords are **strictly mutually disjoint**.

```
               KRAFT'S DISJOINT SUBTREE EXTENSION (l_max = 3)
               
    Depth 0                      ( Root )
                                /        \\
    Depth 1                  [ A ]        ( Node )
                            (l_A=1)       /      \\
                            /     \\    [ B ]    ( Node )
    Depth 2                x       x   (l_B=2)   /      \\
                          / \\     / \\   / \\    [ C ]    [ D ]
    Depth 3              x   x   x   x x   x  (l_C=3)  (l_D=3)
                         \\___________/ \\___/
                          4 leaves at   2 leaves  1 leaf   1 leaf
                            depth 3     depth 3
                            
      Codeword A (len 1) eliminates 2^(3 - 1) = 4 leaves at depth 3.
      Codeword B (len 2) eliminates 2^(3 - 2) = 2 leaves at depth 3.
      Codewords C, D (len 3) each eliminate 2^(3 - 3) = 1 leaf at depth 3.
      Total eliminated: 4 + 2 + 1 + 1 = 8 <= 2^3 = 8 (Budget fully spent!)
```

The total number of leaves dominated by all $K$ codewords cannot exceed the
total available leaves at depth $l_{\max}$:
$$\sum_{i=1}^K 2^{l_{\max} - l_i} \le 2^{l_{\max}}$$

Divide both sides of the inequality by $2^{l_{\max}}$:
$$\sum_{i=1}^K \frac{2^{l_{\max} - l_i}}{2^{l_{\max}}} \le 1$$

$$\sum_{i=1}^K 2^{-l_i} \le 1$$

This is the celebrated **Kraft Inequality**.

### McMillan's Theorem: You Cannot Cheat the Budget with Lookahead
You might wonder: *"What if we abandon instantaneous prefix-free codes and
allow lookahead decoding? Can a non-prefix uniquely decodable code squeeze more
codewords into the same lengths?"*

In 1956, Brockway McMillan proved that **any uniquely decodable code—even one
requiring unbounded lookahead—must still satisfy Kraft's inequality**:
$$\sum_{i=1}^K 2^{-l_i} \le 1$$

**Key Takeaway**: You gain zero compression advantage by abandoning
instantaneous prefix codes. For any uniquely decodable code with lengths
$\{l_i\}$, there exists an instantaneous prefix-free code with the exact same
lengths.

### The Real Estate Metaphor: A Finite Budget of Exactly 1.0
Think of Kraft's inequality as a strict financial currency budget:
* The universe of all possible binary messages has a total currency value of
  **\$1.00**.
* Every codeword you purchase costs $2^{-l_i}$ dollars.

```
                       THE CODEWORD REAL ESTATE BUDGET
                       
     Codeword Length (l_i)    Cost = 2^(-l_i)    Percentage of Entire Tree
     ---------------------    ---------------    -------------------------
     1 bit  ("0")             1/2 = 0.5000       50.00% of all code space
     2 bits ("10")            1/4 = 0.2500       25.00% of all code space
     3 bits ("110")           1/8 = 0.1250       12.50% of all code space
     4 bits ("1110")          1/16 = 0.0625       6.25% of all code space
     8 bits                   1/256 ≈ 0.0039      0.39% of all code space
     16 bits                  1/65536 ≈ 1.5e-5    0.0015% of all code space
```

### Complete vs. Incomplete Codes
* **Complete Code ($\sum 2^{-l_i} = 1$)**: Every internal node in the binary
  tree has exactly two children. The budget is 100% spent. No leaf capacity is
  wasted.
* **Incomplete Code ($\sum 2^{-l_i} < 1$)**: The code has slack. There are
  unassigned branches or internal nodes with only one child. At least one
  codeword could be shortened without violating the prefix-free condition.

---

## 4. Shannon's Source Coding Theorem: The Non-Negotiable Speed Limit

Given an alphabet $\mathcal{X} = \{x_1, \dots, x_K\}$ where each symbol $x_i$
appears with known probability $p_i$, our engineering objective is to minimize
the **expected codeword length**:
$$L = \mathbb{E}[l(X)] = \sum_{i=1}^K p_i l_i$$

subject to the physical Kraft constraint that the code must be decodable:
$$\sum_{i=1}^K 2^{-l_i} \le 1$$

### Derivation of Optimal Lengths via Lagrange Multipliers
To find the theoretical lower bound, we temporarily relax the constraint that
lengths $l_i$ must be integers, treating them as continuous real numbers.

Because expected length $L$ is strictly increasing in each $l_i$, any optimal
code will exhaust the full budget: $\sum 2^{-l_i} = 1$.

We set up the Lagrangian:
$$J(l_1, \dots, l_K, \lambda) = \sum_{i=1}^K p_i l_i$$
$$+ \lambda \left(\sum_{i=1}^K 2^{-l_i} - 1\right)$$

Recalling that $2^{-l_i} = e^{-l_i \ln 2}$, we differentiate with respect to
each $l_i$:
$$\frac{\partial J}{\partial l_i} = p_i - \lambda (\ln 2) 2^{-l_i} = 0$$

Solving for $2^{-l_i}$:
$$2^{-l_i} = \frac{p_i}{\lambda \ln 2}$$

Summing both sides over all $i = 1, \dots, K$:
$$\sum_{i=1}^K 2^{-l_i} = \frac{1}{\lambda \ln 2} \sum_{i=1}^K p_i$$
$$= \frac{1}{\lambda \ln 2}$$

Since the complete budget requires $\sum 2^{-l_i} = 1$, we must have:
$$\lambda \ln 2 = 1$$

Substituting this back into our expression for $2^{-l_i}$:
$$2^{-l_i} = p_i \implies -l_i = \log_2(p_i)$$

$$l_i^* = -\log_2(p_i)$$

### The Profound Equivalence: Optimal Codeword Length = Surprisal
This is one of the most beautiful unifications in all of science:
> **The mathematically optimal length for symbol $x_i$ is exactly its**
> **self-information (surprisal)!**
> $$l_i^* = I(x_i) = \log_2\left(\frac{1}{p_i}\right)$$

Now, let us calculate the expected codeword length under these optimal lengths:
$$L^* = \sum_{i=1}^K p_i l_i^* = -\sum_{i=1}^K p_i \log_2(p_i) = H(X)$$

Claude Shannon did not invent the entropy formula out of thin air. **Shannon**
**entropy $H(X)$ is the exact, unyielding average length of the shortest**
**possible binary description of a random variable.**

### The Integer Constraint & The 1-Bit Penalty
In reality, binary trees cannot have fractional depths. A codeword must consist
of an integer number of bits: $l_i \in \{1, 2, 3, \dots\}$.

If we round each optimal length up to the nearest integer:
$$l_i = \lceil -\log_2 p_i \rceil$$

This choice automatically satisfies Kraft's inequality:
$$\sum_{i=1}^K 2^{-l_i} = \sum_{i=1}^K 2^{-\lceil -\log_2 p_i \rceil}$$
$$\le \sum_{i=1}^K 2^{\log_2 p_i} = \sum_{i=1}^K p_i = 1$$

Because $\lceil x \rceil < x + 1$, each codeword length satisfies:
$$-\log_2 p_i \le l_i < -\log_2 p_i + 1$$

Multiplying by $p_i$ and summing over all symbols:
$$\sum_{i=1}^K p_i (-\log_2 p_i) \le \sum_{i=1}^K p_i l_i$$
$$< \sum_{i=1}^K p_i (-\log_2 p_i + 1)$$

$$H(X) \le L < H(X) + 1$$

### Shannon's Noiseless Source Coding Theorem (1948)
For any discrete memoryless source $X$:
1. **Lower Bound**: No uniquely decodable binary code can achieve an expected
   length smaller than the source entropy: $L \ge H(X)$.
2. **Upper Bound**: There always exists a prefix-free code whose expected length
   is strictly less than $H(X) + 1$ bits per symbol.

### Beating the 1-Bit Penalty via Block Coding
The "+1 bit" overhead arises because we round up fractional bits on every single
symbol. If an alphabet has entropy $H(X) = 1.1\text{ bits}$, but we must assign
at least 1 or 2 integer bits per symbol, we waste up to 0.9 bits per character.

How do we eliminate this rounding waste? **Group symbols into blocks!**

Consider encoding blocks of $n$ independent symbols $X^n = (X_1, \dots, X_n)$.
The entropy of $n$ independent symbols is $H(X^n) = n H(X)$.
Applying Shannon's theorem to the block of length $n$:
$$n H(X) \le L_n < n H(X) + 1$$

Dividing through by the block length $n$:
$$H(X) \le \frac{L_n}{n} < H(X) + \frac{1}{n}$$

As the block size grows ($n \to \infty$), the per-symbol overhead $\frac{1}{n}$
vanishes to zero:
$$\lim_{n \to \infty} \frac{L_n}{n} = H(X)$$

By encoding large blocks of data together, we can compress any memoryless
source arbitrarily close to its Shannon entropy bound!

---

## 5. Huffman Coding: The Bottom-Up Greedy Miracle

Knowing that an optimal prefix code exists is one thing; constructing it
efficiently is another.

### Shannon-Fano Coding and Its Flaw
In 1948–1949, Claude Shannon and Robert Fano independently proposed a **top-down
heuristic**:
1. Sort all symbols by decreasing probability.
2. Divide the list into two subsets whose total probabilities are as close to
   equal as possible.
3. Assign `0` to the first subset and `1` to the second subset.
4. Recursively repeat on each subset until every symbol is isolated.

While intuitive, Shannon-Fano coding is **greedy from the top down**, and it
often gets trapped in suboptimal branch splits.

### The Huffman Breakthrough (1951)
In 1951, David Huffman was a doctoral student at MIT taking Robert Fano's
information theory class. Fano gave students an option: take a traditional
final exam, or solve the open research problem of finding a provably optimal
prefix code.

After months of failed attempts to fix Fano's top-down approach, Huffman
decided to invert the perspective entirely: **build the tree from the bottom up
by pairing the least likely symbols first.**

### Step-by-Step Worked Numerical Example
Let us trace Huffman's algorithm on a 5-symbol source alphabet $\mathcal{X}$:

```
     Symbol       Probability p_i    Ideal Bit Length (-log2 p_i)
     ------       ---------------    ----------------------------
       A               0.40          -log2(0.40) ≈ 1.32 bits
       B               0.25          -log2(0.25) = 2.00 bits
       C               0.20          -log2(0.20) ≈ 2.32 bits
       D               0.10          -log2(0.10) ≈ 3.32 bits
       E               0.05          -log2(0.05) ≈ 4.32 bits
```

#### Step 1: Merge the Two Smallest Nodes
* Smallest probabilities: `E (0.05)` and `D (0.10)`.
* Merge into composite node `[DE]` with combined weight $0.05 + 0.10 = 0.15$.
* Active node pool: `{ [DE]: 0.15, C: 0.20, B: 0.25, A: 0.40 }`.

#### Step 2: Merge the Next Two Smallest Nodes
* Smallest active nodes: `[DE] (0.15)` and `C (0.20)`.
* Merge into composite node `[CDE]` with combined weight $0.15 + 0.20 = 0.35$.
* Active node pool: `{ B: 0.25, [CDE]: 0.35, A: 0.40 }`.

#### Step 3: Merge the Next Two Smallest Nodes
* Smallest active nodes: `B (0.25)` and `[CDE] (0.35)`.
* Merge into composite node `[BCDE]` with weight $0.25 + 0.35 = 0.60$.
* Active node pool: `{ A: 0.40, [BCDE]: 0.60 }`.

#### Step 4: Final Merge to Root
* Merge `A (0.40)` and `[BCDE] (0.60)` into `[Root]` with total weight $1.00$.

```
                       THE COMPLETE HUFFMAN TREE
                       
                                [ Root ] (1.00)
                                 /     \\
                             0  /       \\  1
                               /         \\
                            [ A ]      [ BCDE ] (0.60)
                            (0.40)      /      \\
                                    0  /        \\  1
                                      /          \\
                                   [ B ]       [ CDE ] (0.35)
                                   (0.25)       /     \\
                                            0  /       \\  1
                                              /         \\
                                           [ C ]       [ DE ] (0.15)
                                           (0.20)       /    \\
                                                    0  /      \\  1
                                                      /        \\
                                                   [ D ]      [ E ]
                                                   (0.10)     (0.05)
```

#### Codeword Assignment and Performance Audit
Traversing from Root to each leaf (Left = `0`, Right = `1`):

```
 Symbol   Prob p_i   Codeword   Length l_i   p_i * l_i   Kraft Cost 2^(-l_i)
 ------   --------   --------   ----------   ---------   -------------------
   A        0.40     0              1        0.4000            0.5000
   B        0.25     10             2        0.5000            0.2500
   C        0.20     110            3        0.6000            0.1250
   D        0.10     1110           4        0.4000            0.0625
   E        0.05     1111           4        0.2000            0.0625
 ---------------------------------------------------------------------------
  Total     1.00                             L = 2.1000  Sum = 1.0000
```

Let us audit the performance of this Huffman code:
1. **Expected Length**:
   $$L = \sum_{i=1}^5 p_i l_i = 0.4(1) + 0.25(2) + 0.2(3) + 0.1(4) + 0.05(4)$$
   $$= \mathbf{2.1000\text{ bits/symbol}}$$
2. **Source Entropy**:
   $$H(X) = -\sum_{i=1}^5 p_i \log_2(p_i) \approx \mathbf{2.0414\text{ bits}}$$
3. **Budget Completeness**:
   $$\sum_{i=1}^5 2^{-l_i} = 0.5 + 0.25 + 0.125 + 0.0625 + 0.0625 = 1.0000$$
   The code is 100% complete—not a single leaf of the binary tree is wasted!
4. **Coding Efficiency**:
   $$\eta = \frac{H(X)}{L} = \frac{2.0414}{2.1000} \approx \mathbf{97.21\%}$$
5. **Redundancy Overhead**:
   $$R = L - H(X) = 2.1000 - 2.0414 = \mathbf{0.0586\text{ bits/symbol}}$$
   The Huffman code achieves an expected length within $0.06$ bits of the
   theoretical Shannon limit!

### Why Huffman Coding is Mathematically Optimal
The proof of optimality rests on two structural properties:

1. **Sibling Invariant at Maximum Depth**:
   In any optimal binary prefix tree, the two symbols with the lowest
   probabilities ($p_{K-1}$ and $p_K$) must appear at the deepest level of the
   tree and can be assigned as sibling leaves.
   * *Proof by contradiction*: If they were not at maximum depth, we could swap
     them with deeper nodes having higher probabilities, strictly reducing
     $\sum p_i l_i$.
2. **Inductive Reduction**:
   Let tree $T$ have sibling leaves $x_{K-1}$ and $x_K$ attached to parent
   node $w$. The expected length of $T$ is:
   $$L(T) = \sum_{i=1}^{K-2} p_i l_i + p_{K-1} (l_w + 1) + p_K (l_w + 1)$$
   $$L(T) = L(T') + (p_{K-1} + p_K)$$
   where $T'$ is the reduced tree with $x_{K-1}$ and $x_K$ replaced by their
   composite parent $w$ of weight $p_w = p_{K-1} + p_K$.
   Because $(p_{K-1} + p_K)$ is a constant independent of the tree structure,
   $L(T)$ is minimized if and only if $L(T')$ is minimized.
   By induction on alphabet size $K$, the greedy merge yields the globally
   optimal prefix tree.

### The Limitation of Huffman: The Fractional Bit Problem
Despite its mathematical optimality among prefix trees, Huffman coding suffers
from a fundamental limitation: **it must assign an integer number of bits to
every symbol.**

When a symbol's probability does not match an exact power of two
($p_i \ne 2^{-k}$), the optimal codeword length $-\log_2 p_i$ is a non-integer
fraction. Huffman is forced to round this length to an integer, incurring
redundancy.

#### The Extreme Case: Highly Biased Coin
Consider a source producing independent tosses of a heavily biased coin:
* $P(\text{Heads}) = 0.99$
* $P(\text{Tails}) = 0.01$

The Shannon entropy of this source is:
$$H(p) = -0.99 \log_2(0.99) - 0.01 \log_2(0.01) \approx 0.0808\text{ bits}$$

Each flip conveys less than $0.09$ bits of real information.
However, because any binary code must assign at least 1 bit to distinguish
Heads from Tails, Huffman coding **must assign 1 bit to Heads and 1 bit to
Tails**:
$$L_{\text{Huffman}} = 0.99(1) + 0.01(1) = \mathbf{1.0000\text{ bit}}$$

* Redundancy gap: $1.0000 - 0.0808 = \mathbf{0.9192\text{ bits/symbol}}$.
* Over **$91\%$ of the transmitted bandwidth is pure waste**!

To escape this integer gridlock, modern compression discards one-to-one symbol
trees in favor of **fractional-bit stream encoders**:
* **Arithmetic Coding**: Represents the entire sequence as a single narrowing
  subinterval in $[0, 1)$, allocating exact fractional bits per symbol.
* **Asymmetric Numeral Systems (ANS)**: The breakthrough modern framework that
  encodes fractional bits with the blisteringly fast lookup speed of a finite
  state machine.

---

## 6. Applied Engineering Contexts

The principles of Kraft's inequality and Huffman trees form the backbone of
production systems handling exabytes of data every day:

### 1. DEFLATE: The Engine of gzip, zip, and PNG (RFC 1951)
Every `.tar.gz`, `.zip`, `.png`, and HTTP `Content-Encoding: gzip` payload
relies on the **DEFLATE** algorithm. DEFLATE operates as a two-stage pipeline:
1. **LZ77 Deduplication**: Scans a sliding dictionary buffer (typically 32 KB)
   for repeating byte patterns, replacing duplicates with $\langle\text{length,
   distance}\rangle$ pointer pairs.
2. **Dual Huffman Encoders**: Compresses the resulting stream using two distinct
   Huffman trees:
   * One tree encodes literal bytes ($0$–$255$) plus end-of-block and match
     length codes ($257$–$285$).
   * A second tree encodes match backward distances ($0$–$29$).
3. DEFLATE dynamically chooses between **static pre-shared Huffman tables** (for
   tiny files where transmitting tree metadata would inflate file size) and
   **dynamic custom Huffman trees** computed specifically for each data block.

### 2. HTTP/2 and HTTP/3 HPACK/QPACK (RFC 7541 & RFC 9204)
Modern web browsers transmit dozens of repetitive HTTP headers with every
request (`user-agent`, `accept-encoding`, `cookie`, `:path`).
* To prevent bandwidth bloat and latency spikes on mobile connections, HTTP/2
  introduced **HPACK**.
* HPACK employs a **canonical static Huffman table** trained offline on a
  massive corpus of millions of real-world web requests.
* Because both client and server store this identical static table in memory,
  headers can be compressed instantaneously with zero header table transmission
  overhead.

### 3. Next-Gen Compression: Zstandard & Apple LZFSE
While Huffman coding dominated computing for over 60 years, modern data centers
demand both higher compression density and multi-gigabyte-per-second throughput.
* **Facebook's Zstandard (`zstd`)**: Replaces traditional Huffman with **Finite
  State Entropy (FSE)**, an implementation of Jarek Duda's **Asymmetric Numeral
  Systems (ANS)**. ANS achieves the fractional-bit compression density of
  arithmetic coding while executing at the speed of a table-driven state
  machine.
* **Apple LZFSE**: Deployed across macOS and iOS system updates and asset
  caches, combining LZ-style parsing with Finite State Entropy to maximize
  decompression speed on mobile CPUs.

---

**[Course Home](../README.md) | [Python Lab](../python/02_source_coding_kraft_and_huffman.ipynb) | [Prev: Ch 01: Quantifying Surprise](01_self_information_and_shannon_entropy.ipynb) | [Next: Ch 03: Relative Entropy & KL](03_relative_entropy_and_kl_divergence.ipynb)**